In [ ]:
# Import required libraries
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

**⚠️ Data Requirement — BioImageArchive:** This notebook requires raw microscopy data from BioImageArchive. Download the dataset and set `data_archive_path` to your local BioImageArchive directory (see README).

**Pipeline step 1/5** — must be run before downstream analysis notebooks in this folder.

**Run analysis scripts in this order:**
1. `0_combined_df_ext_H_R.ipynb` — load and combine raw image data
2. `1_fluo_binary_ext_H_R_rep1/2.ipynb` — extract fluorescence binary data (run for each replicate)
3. `1b_generate_masks_rep1/2.ipynb` — generate segmentation masks (run for each replicate)
4. `2_first_colonies_ext_H_R_rep1/2.ipynb` — annotate first-appearance colonies (run for each replicate)
5. `3_first_colonies_ext_H_R_all_rep.ipynb` — calculate regrowth fractions

# Aggregate Fluorescence Intensity Data for ext_H_R Strain

## Overview
This notebook aggregates fluorescence intensity data from multiple experimental replicates and positions for the ext_H_R strain.

## Workflow
1. **Input**: Data comes from `Image_Data/Figure3` study component in BioImageArchive repo
2. **Data Collection**: Iterates over all replicates and positions and loads `single_cell_props.csv` files.
3. **Output**: Saves the combined dataframe as `0_combined_df_ext_H_R.csv`

In [ ]:
# set path to BioImageArchive data directory:
data_archive_path = '/Volumes/ScientificData/Users/Giulia(botgiu00)/Papers/bottacin2025/BioImageArchive/'

# input and output paths relative to BioImageArchive location
input_dir = os.path.join(data_archive_path, 'Image_Data/Figure3/ext_H_R')
output_dir = '0_combined_df_ext_H_R.csv'

# Initialize list to store individual dataframes
all_data = []

# Iterate through replicate folders
for replicate_folder in os.listdir(input_dir):
    replicate_path = os.path.join(input_dir, replicate_folder)
    
    # Process only replicate directories
    if os.path.isdir(replicate_path) and replicate_folder.startswith('replicate'):
        
        # Iterate through position folders within each replicate
        for pos_folder in os.listdir(replicate_path):
            if pos_folder.startswith('pos'):
                pos_path = os.path.join(replicate_path, pos_folder, 'phase', 'fluo_intensities.csv')
                
                # Load data if CSV file exists
                if os.path.isfile(pos_path):
                    df = pd.read_csv(pos_path)
                    
                    # Add metadata columns
                    df.insert(1, 'strain', 'ext_H_R')
                    df.insert(2, 'replicate', replicate_folder)
                    df.insert(3, 'pos', pos_folder)
                    
                    # Append to collection
                    all_data.append(df)

# Combine all dataframes and save to output file
if all_data:
    combined_df_ext_H_R = pd.concat(all_data, ignore_index=True)
    combined_df_ext_H_R.to_csv(output_dir, index=False)
    print(f"Successfully combined {len(all_data)} datasets")
    print(f"Total rows: {len(combined_df_ext_H_R)}")
    print(f"Output saved to: {output_dir}")
else:
    print("Warning: No data files found")

# Display first few rows
combined_df_ext_H_R.head()